# ReefScan — Parts C & D: self-hosted Qwen2.5-VL (vLLM) — run start to finish

**Runtime → L4 or A100, then `Run all`. No restarts, no manual steps.** ~15–20 min total
(install + weight download + benchmark). Part B (the CUDA kernel) is a separate notebook and is
already done — it's not here.

Why no restart is needed: vLLM runs as a **subprocess**, so it uses the freshly-installed packages
directly — this notebook's own kernel never imports torch/vllm, so the mid-run torch swap can't
break it. Every fix we found is baked in (vllm 0.10.2 cu128, transformers 4.55.2, on-disk rope-config
patch, JSON mm-limit, HF-504 retry).

### 1. Install (one-time, ~2–3 min with uv — slow, not frozen)

In [ ]:
!pip install -q uv
# vllm 0.10.2 = cu128 wheel (matches Colab) + Qwen2.5-VL; transformers 4.55.2 (<4.56, vLLM needs
# all_special_tokens_extended); openai/dotenv for the benchmark client.
!uv pip install --system 'vllm==0.10.2' 'transformers==4.55.2' 'openai' 'python-dotenv'
print('installed — NO kernel restart needed (vLLM runs in a subprocess).')

### 2. Clone repo + GPU (via nvidia-smi — we never import torch here)

In [ ]:
import subprocess
!git clone -q https://github.com/HrishiKabra/reefscan.git 2>/dev/null || echo 'repo already present'
%cd /content/reefscan
GPU = subprocess.run(['nvidia-smi','--query-gpu=name','--format=csv,noheader'],
                     capture_output=True, text=True).stdout.strip().splitlines()[0]
print('GPU:', GPU)

### 3. Patch Qwen's rope config on disk + start the vLLM server
Qwen2.5-VL's config carries conflicting rope fields; we rewrite `rope_scaling` to one clean form so
vLLM accepts it, then serve from the local path (`--served-model-name` keeps the API id stable).

In [ ]:
import sys, time, os, json, urllib.request
from huggingface_hub import snapshot_download

def _get_model():
    for a in range(6):
        try: return snapshot_download('Qwen/Qwen2.5-VL-7B-Instruct')
        except Exception as e:
            print(f'HF fetch attempt {a+1} failed ({type(e).__name__}) — transient, retrying in 10s...'); time.sleep(10)
    return snapshot_download('Qwen/Qwen2.5-VL-7B-Instruct', local_files_only=True)

path = _get_model()
cfg_path = os.path.join(path, 'config.json')
cfg = json.load(open(cfg_path))
_ms = cfg.get('rope_scaling', {}).get('mrope_section', [16, 24, 24])  # preserve the model's own
cfg['rope_scaling'] = {'rope_type': 'mrope', 'mrope_section': _ms}
json.dump(cfg, open(cfg_path, 'w'))
print('patched rope_scaling ->', cfg['rope_scaling'])

os.makedirs('logs', exist_ok=True)
srv = subprocess.Popen(
    [sys.executable, '-m', 'vllm.entrypoints.openai.api_server',
     '--model', path,
     '--served-model-name', 'Qwen/Qwen2.5-VL-7B-Instruct',
     '--gpu-memory-utilization', '0.9',
     '--max-model-len', '8192',
     '--max-logprobs', '20',
     '--limit-mm-per-prompt', '{"image": 1}',
     '--port', '8000'],
    stdout=open('logs/vllm.log','w'), stderr=subprocess.STDOUT)
print('starting vLLM (loads ~16GB; a few minutes)...')
up = False
for i in range(240):
    try: urllib.request.urlopen('http://localhost:8000/v1/models', timeout=2); up = True; break
    except Exception: time.sleep(5)
print('vLLM server UP after ~%ds' % (i*5) if up else 'NOT up yet — see logs/vllm.log below')
!tail -8 logs/vllm.log

### 3b. Sanity — does Qwen actually SEE the image?
If WHITE and GREEN come back as different colors, the vision path works and the benchmark number is trustworthy.

In [ ]:
from openai import OpenAI
import numpy as np
from PIL import Image
_cl = OpenAI(base_url='http://localhost:8000/v1', api_key='EMPTY')
def _ask(arr):
    b = io.BytesIO(); Image.fromarray(arr).save(b, 'PNG')
    u = 'data:image/png;base64,' + __import__('base64').b64encode(b.getvalue()).decode()
    content = [{'type':'text','text':'What color is this image? One word.'},
               {'type':'image_url','image_url':{'url':u}}]
    r = _cl.chat.completions.create(model='Qwen/Qwen2.5-VL-7B-Instruct', max_tokens=10,
        temperature=0, messages=[{'role':'user','content':content}])
    return r.choices[0].message.content
print('WHITE ->', _ask(np.full((224,224,3),255,np.uint8)))
print('GREEN ->', _ask(np.dstack([np.zeros((224,224)),np.full((224,224),180),np.zeros((224,224))]).astype(np.uint8)))

### 4. Benchmark Qwen on the NOAA test split (forced A/B + logprobs → acc / macro-F1 / ECE)
`--limit 300` for a fast first pass; drop it for the full 1,565 (resumable JSONL cache).

In [ ]:
!python -m backend.vlm_benchmark --model Qwen/Qwen2.5-VL-7B-Instruct \
    --base-url http://localhost:8000/v1 --api-key EMPTY --workers 16 --limit 300

### 5. Three-way comparison — DINOv2 specialist vs GPT-4o vs Qwen2.5-VL

In [ ]:
import glob
from pathlib import Path
E = Path('docs/eval'); bym = {}
spec = json.loads((E/'metrics.json').read_text())
bym['DINOv2-B specialist'] = (spec['accuracy'], spec['macro_f1'], spec.get('ece'))
cands = list(glob.glob(str(E/'vlm_benchmark_*.json')))
if (E/'vlm_benchmark.json').exists(): cands.append(str(E/'vlm_benchmark.json'))
for f in cands:
    d = json.loads(open(f).read()); bym[d['model']] = (d['accuracy'], d['macro_f1'], d.get('ece'))
print(f"{'model':<32}{'accuracy':>9}{'macro-F1':>10}{'ECE':>8}"); print('-'*59)
for m,(a,f1,e) in bym.items():
    es = f'{e:.4f}' if e is not None else '  n/a'
    print(f'{m:<32}{a:>9.4f}{f1:>10.4f}{es:>8}')

### 6. Serving sweep — p50/p95/p99 latency + throughput vs concurrency (1→32)
Async sweep of the real image workload against the running server (vLLM 0.10.2 has no
`vllm bench serve` subcommand, so we measure the endpoint directly). Writes `docs/eval/vllm_serving_sweep.*`.

In [ ]:
import asyncio, io, base64, httpx
import numpy as np
from PIL import Image
LEVELS = [1,4,8,16,32]
def _p(v,q):
    if not v: return 0.0
    v=sorted(v); k=(len(v)-1)*q/100; f=int(k)
    return round(v[f]+(v[min(f+1,len(v)-1)]-v[f])*(k-f),1)
arr = np.random.default_rng(0).integers(0,255,(224,224,3),dtype=np.uint8)
buf = io.BytesIO(); Image.fromarray(arr).save(buf,format='PNG'); b64 = base64.b64encode(buf.getvalue()).decode()
body = {'model':'Qwen/Qwen2.5-VL-7B-Instruct','max_tokens':1,'temperature':0,
    'messages':[{'role':'user','content':[{'type':'text','text':'A for healthy, B for bleached. One letter.'},
    {'type':'image_url','image_url':{'url':'data:image/png;base64,'+b64}}]}]}
async def _one(cl):
    t=time.perf_counter(); r=await cl.post('http://localhost:8000/v1/chat/completions',json=body); r.raise_for_status()
    return (time.perf_counter()-t)*1000
async def _level(c,n=48):
    sem=asyncio.Semaphore(c); lat=[]
    async with httpx.AsyncClient(timeout=120) as cl:
        async def w():
            async with sem: lat.append(await _one(cl))
        t=time.perf_counter(); await asyncio.gather(*[w() for _ in range(n)]); wall=time.perf_counter()-t
    return {'concurrency':c,'p50':_p(lat,50),'p95':_p(lat,95),'p99':_p(lat,99),'throughput':round(n/wall,2)}
rows = [await _level(c) for c in LEVELS]
print(f'[{GPU}]  conc   p50 ms   p95 ms   p99 ms   req/s')
for r in rows: print(f"      {r['concurrency']:>4} {r['p50']:>8} {r['p95']:>8} {r['p99']:>8} {r['throughput']:>7}")
json.dump({'gpu':GPU,'rows':rows}, open('docs/eval/vllm_serving_sweep.json','w'), indent=2)

In [ ]:
import matplotlib.pyplot as plt
cs=[r['concurrency'] for r in rows]
fig,(a1,a2)=plt.subplots(1,2,figsize=(12,4.5))
for k,col in [('p50','#2166ac'),('p95','#b2182b'),('p99','#762a83')]:
    a1.plot(cs,[r[k] for r in rows],'-o',label=k,color=col)
a1.set_xlabel('concurrency'); a1.set_ylabel('latency (ms)'); a1.set_title(f'Qwen2.5-VL serving latency ({GPU})'); a1.legend(); a1.grid(alpha=.25)
a2.plot(cs,[r['throughput'] for r in rows],'-o',color='#1b7837')
a2.set_xlabel('concurrency'); a2.set_ylabel('throughput (req/s)'); a2.set_title('Throughput'); a2.grid(alpha=.25)
fig.tight_layout(); fig.savefig('docs/eval/vllm_serving_sweep.png',dpi=140,bbox_inches='tight'); plt.show()
print('paste the table + this figure back to commit them.')